# Projeto Final — Fundamentos de Machine Learning

## Regressão e Classificação Binária: do problema à avaliação do modelo

Neste projeto você vai percorrer **todo o ciclo de um projeto de Machine Learning supervisionado**, desde a definição do problema até a análise crítica dos resultados, usando um **dataset de sua escolha**.

> **Lembre-se:** mais importante do que a complexidade do modelo é a **aplicação correta dos fundamentos** vistos na disciplina. Uma regressão linear bem validada, sem vazamento de informação e bem interpretada vale mais do que um modelo sofisticado avaliado de forma errada.

### Escopo

| Incluído | Fora do escopo |
|---|---|
| **Regressão** (alvo numérico contínuo) | Classificação multiclasse |
| **Classificação binária** (alvo com 2 classes) | Clusterização / aprendizado não supervisionado |
| Pré-processamento, validação, overfitting, regularização, classes desbalanceadas, vazamento de informação | Redes neurais / deep learning |

> Se o alvo do seu dataset tiver mais de duas classes, você pode **transformá-lo em binário** (ex.: qualidade do vinho ≥ 7 → "bom", caso contrário "não bom"). Justifique essa decisão na Etapa 1.

### Como usar este notebook

- Ele é um **roteiro**. Cada etapa descreve **o que** deve ser feito, traz **dicas** de funções úteis e **perguntas** que você deve responder.
- **Você decide como implementar.** As células `# Seu código aqui` são espaços para o seu código; adicione quantas células precisar.
- Algumas etapas trazem **funções prontas para gerar gráficos**. Elas são **sugestões** para você não perder tempo com formatação; use-as, adapte-as ou crie seus próprios gráficos.
- Responda às perguntas **✍️ Responda** com suas palavras, nas células de texto indicadas.

### Entrega

- Notebook `.ipynb` (Jupyter ou Google Colab) **executado do início ao fim, sem erros**, com as saídas visíveis — ou a exportação em PDF do notebook executado.
- Inclua o link ou o arquivo do dataset (ou o código que o gera, no caso de dados sintéticos).
- Prazo: **19 / 11 / 2026**
- Bibliotecas: `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`.

Os critérios de avaliação estão no **Apêndice B**. Leia-os antes de começar.

## Identificação

- **Nome:**
- **Dataset escolhido:**
- **Link / fonte do dataset:**
- **Tipo de problema:** ( ) Regressão  ( ) Classificação binária

### Bibliotecas

A célula abaixo importa as bibliotecas básicas. Importe o que mais precisar do `scikit-learn` ao longo do projeto.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42  # use esta semente em tudo que for aleatório, para que os resultados possam ser reproduzidos

---
# Etapa 1 — Definição do problema

Antes de escrever qualquer linha de código, é preciso entender **o que** queremos prever e **por quê**. Sugestões de datasets estão no **Apêndice C**.

### ✍️ Responda

1. **Contexto:** qual é o domínio do problema? Quem usaria o modelo e para tomar qual decisão?
2. **Variável alvo:** qual coluna você quer prever? O que ela significa?
3. **Tipo de tarefa:** é regressão ou classificação binária? Por que se trata de aprendizado **supervisionado**?
4. **Unidade de observação:** o que cada linha representa (um cliente, uma casa, um paciente...)?
5. **Momento da previsão:** em que momento o modelo seria usado na prática? Quais informações **estariam disponíveis** nesse momento? *(Essa é a chave para evitar vazamento de informação — volte a ela na Etapa 2.)*
6. **Sucesso:** o que seria um "bom" resultado? Que tipo de erro é mais grave nesse contexto (errar para mais ou para menos; falso positivo ou falso negativo)?

**Suas respostas:**

1.
2.
3.
4.
5.
6.

---
# Etapa 2 — Carregamento, inspeção e limpeza inicial

### O que fazer

1. Carregue o dataset e verifique: número de linhas e colunas, tipos de cada coluna, estatísticas descritivas e **quantidade/percentual de valores ausentes** por coluna.
2. Faça a **limpeza que pode ser feita antes da divisão treino/teste** — operações que **não usam nenhuma estatística calculada a partir dos dados**:
   - remover linhas **duplicadas** e linhas com **alvo ausente**;
   - remover colunas de **identificação** (IDs, nomes, protocolos), que não generalizam;
   - remover colunas que **vazam o alvo** — informações que só existiriam *depois* do evento a prever (reveja a resposta 5 da Etapa 1);
   - corrigir tipos (número lido como texto) e padronizar rótulos (`"sim"`, `"Sim"`, `"SIM"` → `"sim"`).
3. Prepare o alvo: numérico na regressão; **0/1** na classificação, com `1` = classe de interesse (positiva).
4. Separe as features (`X`) do alvo (`y`).

> ⚠️ **Não** faça aqui imputação de ausentes, normalização nem codificação de variáveis. Essas operações **aprendem algo com os dados** e devem ser ajustadas **apenas no treino** (Etapa 5).

**Dicas:** `pd.read_csv`, `df.info()`, `df.describe()`, `df.isna().mean()`, `df.duplicated()`, `df.drop(columns=...)`, `df.dropna(subset=...)`.

In [ ]:
# Seu código aqui

### ✍️ Responda

1. Quantas linhas e colunas sobraram após a limpeza? Quais colunas foram descartadas e por quê?
2. Alguma coluna parecia **suspeita** de vazar informação do alvo? Como você decidiu mantê-la ou removê-la?
3. **(Classificação)** Qual é a proporção de cada classe? As classes são desbalanceadas?

**Suas respostas:**

1.
2.
3.

---
# Etapa 3 — Divisão treino / teste

O conjunto de **teste** simula dados que o modelo nunca viu. Ele deve ser separado **antes** da análise exploratória detalhada e de qualquer pré-processamento, e só será usado **uma única vez**, na Etapa 9. Todas as decisões (transformações, modelo, regularização, limiar) serão tomadas **olhando apenas para o treino**.

### O que fazer

- Separe de **20% a 25%** dos dados para teste, usando `RANDOM_STATE`.
- Em **classificação**, mantenha a mesma proporção de classes no treino e no teste.
- Se os dados tiverem **ordem temporal** e o objetivo for prever o futuro, a divisão deve respeitar o tempo (treino = passado, teste = futuro). Comente se for o seu caso.

**Dicas:** `sklearn.model_selection.train_test_split` (veja o parâmetro `stratify`).

In [ ]:
# Seu código aqui

### ✍️ Responda

1. Por que o conjunto de teste precisa ser separado **antes** da análise exploratória e do pré-processamento?
2. **(Classificação)** O que poderia acontecer com a proporção de classes no teste se você **não** estratificasse a divisão?

**Suas respostas:**

1.
2.

---
# Etapa 4 — Análise exploratória (somente no treino)

A análise exploratória serve para **entender os dados** e **embasar as decisões** de pré-processamento e modelagem. Use **apenas o conjunto de treino**.

### O que fazer (no mínimo)

1. Analise a **variável alvo** (histograma na regressão; contagem de classes na classificação).
2. Analise as **variáveis numéricas**: distribuição de valores, valores extremos, escalas.
3. Analise as **variáveis categóricas**: número de categorias, categorias raras.
4. Verifique a **correlação** entre as numéricas e delas com o alvo.
5. Escolha **ao menos 2 features** que você considera importantes e mostre graficamente a relação de cada uma com o alvo.

> ⚠️ Colunas como CEP, código de região ou "escolaridade de 1 a 5" podem estar armazenadas como números, mas representar **categorias**. Decida como tratá-las.

### Funções sugeridas para gráficos

In [ ]:
def plot_alvo(y, tipo_problema):
    # tipo_problema: "regressao" ou "classificacao"
    fig, ax = plt.subplots(figsize=(7, 4))
    if tipo_problema == "classificacao":
        sns.countplot(x=y, ax=ax)
        ax.set_title("Contagem de cada classe")
    else:
        sns.histplot(y, bins=40, kde=True, ax=ax)
        ax.set_title("Histograma do alvo")
    plt.tight_layout()
    plt.show()


def plot_histogramas(df, colunas):
    n_linhas = int(np.ceil(len(colunas) / 4))
    df[colunas].hist(bins=30, figsize=(14, 3 * n_linhas), layout=(n_linhas, 4))
    plt.tight_layout()
    plt.show()


def plot_correlacao(df, colunas):
    corr = df[colunas].corr()
    tamanho = min(1 + 0.7 * len(colunas), 14)
    plt.figure(figsize=(tamanho, tamanho * 0.85))
    sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1,
                annot=len(colunas) <= 12, fmt=".2f")
    plt.title("Matriz de correlação")
    plt.show()


def plot_feature_vs_alvo(df, feature, alvo, tipo_problema):
    # Escolhe o gráfico conforme o tipo da feature e do problema.
    fig, ax = plt.subplots(figsize=(7, 4))
    numerica = pd.api.types.is_numeric_dtype(df[feature])
    if numerica and tipo_problema == "regressao":
        sns.scatterplot(data=df, x=feature, y=alvo, alpha=0.4, ax=ax)
    elif numerica:
        sns.boxplot(data=df, x=alvo, y=feature, ax=ax)
    elif tipo_problema == "regressao":
        sns.boxplot(data=df, x=feature, y=alvo, ax=ax)
        ax.tick_params(axis="x", rotation=45)
    else:
        taxa = df.groupby(feature)[alvo].mean().sort_values()
        taxa.plot.barh(ax=ax)
        ax.set_xlabel(f"proporção da classe positiva ({alvo} = 1)")
    ax.set_title(f"{feature} x {alvo}")
    plt.tight_layout()
    plt.show()


# Exemplos de uso (df_treino = X_train com a coluna do alvo):
# plot_alvo(y_train, "classificacao")
# plot_histogramas(df_treino, colunas_numericas)
# plot_correlacao(df_treino, colunas_numericas + ["alvo"])
# plot_feature_vs_alvo(df_treino, "idade", "alvo", "classificacao")

In [ ]:
# Seu código aqui

### ✍️ Responda

1. Como se comporta o alvo? **(Regressão)** É muito assimétrico ou tem valores extremos? **(Classificação)** Qual o grau de desbalanceamento?
2. Quais colunas têm valores ausentes e em que proporção? Que estratégia você pretende usar para cada uma (descartar a coluna, imputar com média, mediana, categoria mais frequente...)? Justifique.
3. Há variáveis em **escalas muito diferentes**? Por que isso importa para modelos lineares regularizados (Ridge, Lasso, Regressão Logística)?
4. Há variáveis categóricas com **muitas categorias**? Como vai tratá-las?
5. Quais features parecem mais relacionadas com o alvo? Há pares de features muito correlacionados entre si?
6. Algo **suspeito** nos dados (valores impossíveis, feature com correlação quase perfeita com o alvo, outliers extremos)? O que você fez a respeito?

**Suas respostas:**

1.
2.
3.
4.
5.
6.

---
# Etapa 5 — Pré-processamento

Toda transformação que **aprende algo com os dados** — a média usada na imputação, a média e o desvio padrão da padronização, as categorias conhecidas pelo one-hot encoding — deve ser ajustada **somente nos dados de treino** e depois **aplicada** à validação e ao teste. Caso contrário, informação desses conjuntos "vaza" para o modelo e a avaliação fica otimista demais.

### O que fazer

1. Defina o tratamento de **valores ausentes** (numéricas e categóricas).
2. Defina o **escalonamento** das variáveis numéricas.
3. Defina a **codificação** das variáveis categóricas.
4. *(Opcional)* Transformações adicionais justificadas pela análise exploratória: logaritmo em variável muito assimétrica, criação de novas features, agrupamento de categorias raras.
5. Garanta que tudo seja ajustado **apenas no treino**.

**Dicas:**
- `SimpleImputer`, `StandardScaler`, `MinMaxScaler`, `OneHotEncoder(handle_unknown="ignore")`.
- **Recomendado:** organize o pré-processamento com `ColumnTransformer` e junte-o ao modelo em um `Pipeline`. Assim, quando você usar validação cruzada (Etapa 6), o ajuste do pré-processamento acontece automaticamente **só na parte de treino de cada fold**, sem que você precise se preocupar com isso.

```
                ┌──────────────────────── Pipeline ────────────────────────┐
X (bruto)  ──▶  │  ColumnTransformer                                       │
                │    ├─ numéricas:   imputação ─▶ escalonamento            │ ──▶ previsão
                │    └─ categóricas: imputação ─▶ one-hot encoding         │
                │  ──▶ modelo                                              │
                └──────────────────────────────────────────────────────────┘
```

In [ ]:
# Seu código aqui

### ✍️ Responda

1. Qual estratégia de imputação você usou para as variáveis numéricas? Por que ela é adequada aos seus dados (pense em valores extremos)?
2. Explique, com suas palavras, por que **ajustar o escalonador no dataset inteiro antes da divisão** é vazamento de informação.
3. Quantas colunas existiam antes e depois do pré-processamento? De onde veio a diferença?
4. Se fez transformações adicionais, como garantiu que elas não vazam informação?

**Suas respostas:**

1.
2.
3.
4.

---
# Etapa 6 — Métrica, validação e baseline

### 6.1 Escolha da métrica principal

| Regressão | Quando usar |
|---|---|
| **RMSE** | Mesma unidade do alvo; penaliza mais os erros grandes |
| **MAE** | Mesma unidade do alvo; mais robusto a valores extremos |
| **R²** | Fração da variação do alvo explicada pelo modelo; útil para comparar, mas não diz o tamanho do erro |

| Classificação binária | Quando usar |
|---|---|
| **Acurácia** | Só é informativa quando as classes são **equilibradas** |
| **Precisão** | Quando **falsos positivos** são caros |
| **Recall** | Quando **falsos negativos** são caros |
| **F1** | Equilíbrio entre precisão e recall |
| **ROC-AUC** | Capacidade do modelo de dar probabilidade maior aos positivos do que aos negativos, independente do limiar |

Reporte **várias métricas**, mas escolha **uma principal** para tomar decisões.

### 6.2 Estratégia de validação

Use **validação cruzada (k = 5)** no conjunto de treino para comparar modelos. Registre o desempenho **no treino e na validação** de cada fold: a diferença entre os dois é o seu principal sinal de **overfitting** na Etapa 7.

### 6.3 Baseline

Treine e avalie um **baseline** — um modelo que **não aprende nada** com as features (prevê sempre a média, na regressão, ou sempre a classe mais frequente, na classificação). Todo modelo seu precisa ser **claramente melhor** que ele.

**Dicas:** `KFold` / `StratifiedKFold`, `cross_validate` (veja os parâmetros `scoring` e `return_train_score`), `DummyRegressor`, `DummyClassifier`. No `scikit-learn`, métricas de erro aparecem com sinal negativo (`"neg_root_mean_squared_error"`), porque ele sempre trata "maior" como "melhor".

In [ ]:
# Seu código aqui

### ✍️ Responda

1. Qual métrica principal você escolheu e por quê? Relacione com o tipo de erro mais grave no seu problema.
2. Por que comparar modelos com validação cruzada é mais confiável do que com uma única divisão treino/validação?
3. Qual o desempenho do baseline? **(Classificação)** Por que a acurácia do baseline pode ser enganosamente alta quando as classes são desbalanceadas?

**Suas respostas:**

1.
2.
3.

---
# Etapa 7 — Modelagem, overfitting e regularização

### 7.1 Comparação de modelos

Treine e compare, com a validação cruzada da Etapa 6, **pelo menos três modelos** além do baseline:

- **Regressão:** `LinearRegression`, `Ridge` e `Lasso` (obrigatórios). Opcionalmente, outro algoritmo visto em aula.
- **Classificação:** `LogisticRegression` sem ajuste de pesos, `LogisticRegression` com `class_weight="balanced"` e um terceiro à sua escolha (outro valor de `C` ou outro algoritmo visto em aula).

Monte uma **tabela** com a métrica principal no **treino** e na **validação** (média e desvio padrão entre os folds) para cada modelo.

### 7.2 Efeito da regularização

Escolha um modelo regularizado e **varie manualmente** a intensidade da regularização — `alpha` no Ridge/Lasso ou `C` na Regressão Logística — em pelo menos **7 valores** espaçados em escala logarítmica (ex.: `0.001, 0.01, 0.1, 1, 10, 100, 1000`). Para cada valor, registre a métrica no treino e na validação e faça o gráfico.

> Lembre: no Ridge/Lasso, **`alpha` maior = mais regularização**. Na `LogisticRegression`, **`C` é o inverso**: `C` menor = mais regularização.

| Situação | Treino | Validação | Diagnóstico |
|---|---|---|---|
| A | ruim | ruim | **underfitting** — modelo simples demais (ou regularização forte demais) |
| B | muito bom | bem pior que o treino | **overfitting** — o modelo decorou o treino |
| C | bom | próximo do treino | bom equilíbrio |

### Função sugerida para gráfico

In [ ]:
def plot_treino_vs_validacao(valores, scores_treino, scores_val, nome_parametro, nome_metrica, escala_log=True):
    # valores: lista com os valores do hiperparâmetro testados (ex.: alphas)
    # scores_treino / scores_val: listas com a métrica média no treino e na validação para cada valor
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(valores, scores_treino, "o-", label="treino")
    ax.plot(valores, scores_val, "o-", label="validação")
    if escala_log:
        ax.set_xscale("log")
    ax.set_xlabel(nome_parametro)
    ax.set_ylabel(nome_metrica)
    ax.set_title(f"{nome_metrica} no treino e na validação para cada {nome_parametro}")
    ax.legend()
    plt.tight_layout()
    plt.show()


# Exemplo de uso:
# plot_treino_vs_validacao([0.001, 0.01, 0.1, 1, 10, 100, 1000], rmse_treino, rmse_val, "alpha", "RMSE")

In [ ]:
# Seu código aqui

### ✍️ Responda

1. Todos os modelos superaram o baseline? Por quanto?
2. Algum modelo apresenta **overfitting** ou **underfitting**? Justifique com a diferença entre treino e validação.
3. No gráfico de regularização, o que acontece com treino e validação quando a regularização fica **muito fraca**? E **muito forte**? Qual valor você escolheu e por quê?
4. **(Regressão)** O Lasso zerou algum coeficiente? O que isso significa?
5. **(Classificação)** O que mudou nas métricas ao usar `class_weight="balanced"`? Por quê?
6. O desvio padrão entre os folds é pequeno ou grande em relação à diferença entre os modelos? A diferença entre o melhor e o segundo melhor modelo é confiável?
7. Qual é o seu **modelo final** e por quê? (Considere desempenho, estabilidade e simplicidade.)

**Suas respostas:**

1.
2.
3.
4.
5.
6.
7.

---
# Etapa 8 — Análise dos erros (ainda sem usar o teste)

Antes de ir ao teste, analise **onde** o modelo final erra. Para isso, gere previsões **de validação** para todo o treino: cada linha deve receber a previsão de um modelo que **não** a viu durante o ajuste.

**Dica:** `cross_val_predict` faz exatamente isso (use `method="predict_proba"` para obter probabilidades na classificação).

### O que fazer

- **Regressão:** gráfico real × previsto e gráfico de resíduos.
- **Classificação:** matriz de confusão. *(Opcional)* Avalie se um **limiar** diferente de 0,5 seria mais adequado ao seu problema — o limiar também é uma decisão de modelagem, então deve ser escolhido aqui, **nunca** com o teste.

### Funções sugeridas para gráficos

In [ ]:
def plot_real_vs_previsto(y_real, y_previsto, titulo="Real x previsto"):
    y_real, y_previsto = np.asarray(y_real), np.asarray(y_previsto)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(y_real, y_previsto, alpha=0.4)
    limites = [min(y_real.min(), y_previsto.min()), max(y_real.max(), y_previsto.max())]
    ax.plot(limites, limites, "r--", label="previsão perfeita")
    ax.set_xlabel("valor real")
    ax.set_ylabel("valor previsto")
    ax.set_title(titulo)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_residuos(y_real, y_previsto, titulo="Resíduos"):
    y_real, y_previsto = np.asarray(y_real), np.asarray(y_previsto)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(y_previsto, y_real - y_previsto, alpha=0.4)
    ax.axhline(0, color="r", linestyle="--")
    ax.set_xlabel("valor previsto")
    ax.set_ylabel("resíduo (real - previsto)")
    ax.set_title(titulo)
    plt.tight_layout()
    plt.show()


def plot_matriz_confusao(y_real, y_previsto, titulo="Matriz de confusão"):
    metrics.ConfusionMatrixDisplay.from_predictions(y_real, y_previsto, cmap="Blues")
    plt.title(titulo)
    plt.grid(False)
    plt.show()


def plot_metricas_por_limiar(y_real, probabilidades):
    # Mostra como precisão, recall e F1 mudam conforme o limiar de decisão.
    limiares = np.round(np.arange(0.05, 0.96, 0.05), 2)
    tabela = pd.DataFrame({
        "precisao": [metrics.precision_score(y_real, probabilidades >= t, zero_division=0) for t in limiares],
        "recall": [metrics.recall_score(y_real, probabilidades >= t) for t in limiares],
        "f1": [metrics.f1_score(y_real, probabilidades >= t) for t in limiares],
    }, index=pd.Index(limiares, name="limiar"))
    tabela.plot(figsize=(8, 4.5), marker="o", title="Métricas por limiar de decisão")
    plt.tight_layout()
    plt.show()
    return tabela.round(3)


# Exemplos de uso:
# plot_real_vs_previsto(y_train, previsoes_validacao)
# plot_residuos(y_train, previsoes_validacao)
# plot_matriz_confusao(y_train, previsoes_validacao)
# plot_metricas_por_limiar(y_train, probabilidades_validacao)

In [ ]:
# Seu código aqui

### ✍️ Responda

**Regressão**
1. Os pontos do gráfico real × previsto estão próximos da diagonal? Em que faixa de valores o modelo erra mais?
2. Os resíduos estão espalhados aleatoriamente em torno de zero ou existe algum padrão (formato de funil, de curva)? O que um padrão indicaria?

**Classificação**
1. Quantos falsos positivos e falsos negativos o modelo comete? O que cada um significa no contexto do seu problema?
2. *(Se analisou o limiar)* Qual limiar você escolheu e qual o efeito dele na precisão e no recall em relação a 0,5?

**Suas respostas:**

---
# Etapa 9 — Avaliação final no conjunto de teste

Chegou a hora de usar o teste — **uma única vez**.

### O que fazer

1. Treine o modelo final com **todo o conjunto de treino**.
2. Avalie no **teste** com as mesmas métricas usadas na validação (e com o limiar escolhido, na classificação).
3. Compare com o resultado da validação cruzada e com o baseline.
4. Faça os gráficos de avaliação: **real × previsto** (regressão) ou **matriz de confusão e curva ROC** (classificação).
5. Interprete o modelo: se for linear, mostre os **coeficientes** mais importantes.

> ⚠️ Se, depois de ver o resultado no teste, você voltar e mudar o modelo para melhorar esse número, o teste deixa de ser uma estimativa honesta. Se isso acontecer, **declare** no relatório.

**Dica:** com `Pipeline`, os nomes das features após o pré-processamento podem ser obtidos com `pipeline.named_steps["<nome_da_etapa>"].get_feature_names_out()`, e os coeficientes com `.coef_` no modelo.

### Funções sugeridas para gráficos

In [ ]:
def plot_curva_roc(y_real, probabilidades, titulo="Curva ROC"):
    fig, ax = plt.subplots(figsize=(6, 5))
    metrics.RocCurveDisplay.from_predictions(y_real, probabilidades, ax=ax)
    ax.plot([0, 1], [0, 1], "k--", label="aleatório (AUC = 0,5)")
    ax.set_title(titulo)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


def plot_coeficientes(nomes_features, coeficientes, top=15):
    # Mostra os `top` coeficientes de maior valor absoluto (modelos lineares).
    coefs = pd.Series(np.ravel(coeficientes), index=nomes_features)
    coefs = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(top)
    cores = ["tab:blue" if c > 0 else "tab:red" for c in coefs.values]
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(coefs) + 1))
    ax.barh(coefs.index[::-1], coefs.values[::-1], color=cores[::-1])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{len(coefs)} maiores coeficientes (azul = positivo, vermelho = negativo)")
    plt.tight_layout()
    plt.show()


# Exemplos de uso:
# plot_real_vs_previsto(y_test, previsoes_teste, "Real x previsto — teste")
# plot_matriz_confusao(y_test, previsoes_teste, "Matriz de confusão — teste")
# plot_curva_roc(y_test, probabilidades_teste, "Curva ROC — teste")
# plot_coeficientes(nomes, modelo.coef_)

In [ ]:
# Seu código aqui

### ✍️ Responda

1. O desempenho no teste ficou próximo ao da validação cruzada? Se houve diferença grande, quais podem ser as causas?
2. Quanto o modelo melhorou em relação ao baseline? Essa melhora é relevante **na prática**? Traduza em termos do problema (ex.: "erro médio de R$ 12 mil no preço de um imóvel", "de cada 10 clientes que cancelam, o modelo identifica 7").
3. **(Classificação)** Interprete o valor da ROC-AUC no teste.
4. Quais features têm maior peso no modelo? Isso faz sentido com a análise exploratória e com o conhecimento do domínio?

**Suas respostas:**

1.
2.
3.
4.

---
# Etapa 10 — Conclusões

Escreva um texto curto (1 a 3 parágrafos) como se fosse apresentar o resultado para quem vai usar o modelo:

1. **Resumo:** qual era o problema, qual modelo foi escolhido e qual desempenho atingiu no teste.
2. **Boas práticas aplicadas:** como evitou vazamento de informação, como validou o modelo, como lidou com overfitting e, se for o caso, com o desbalanceamento das classes.
3. **Limitações:** o que pode fazer o modelo falhar na prática (poucos dados, viés na coleta, features ausentes, mudanças ao longo do tempo...)?
4. **Próximos passos:** o que faria com mais tempo ou mais dados?

**Suas conclusões:**

---
# Apêndice A — Checklist antes de entregar

Marque cada item (troque `[ ]` por `[x]`):

**Definição e dados**
- [ ] O problema, o alvo e o momento da previsão estão descritos (Etapa 1).
- [ ] O alvo é numérico (regressão) ou tem exatamente duas classes (classificação).
- [ ] Removi IDs e qualquer coluna que só seria conhecida **depois** do evento a prever.

**Vazamento de informação**
- [ ] Separei o teste **antes** da análise exploratória e do pré-processamento.
- [ ] Imputação, escalonamento e codificação são ajustados **só no treino** (de preferência dentro de um `Pipeline`).
- [ ] Regularização, modelo final e limiar foram escolhidos com validação cruzada no treino, **nunca** com o teste.
- [ ] O conjunto de teste foi usado **uma única vez**, na Etapa 9.

**Avaliação e overfitting**
- [ ] Comparei todos os modelos com um **baseline**.
- [ ] A métrica principal está justificada pelo problema.
- [ ] Comparei desempenho de treino e de validação e discuti overfitting/underfitting.
- [ ] Variei a regularização e analisei seu efeito.
- [ ] **(Classificação)** Estratifiquei a divisão, avaliei o impacto do desbalanceamento e não me baseei só na acurácia.

**Entrega**
- [ ] O notebook roda do início ao fim sem erros (*Kernel → Restart & Run All*).
- [ ] Todas as perguntas **✍️ Responda** foram respondidas.
- [ ] O dataset (ou o link para ele) está incluído.

# Apêndice B — Critérios de avaliação

| Critério | O que será observado | Peso |
|---|---|---|
| **1. Definição do problema** | Contexto claro, alvo e tipo de tarefa corretos, reflexão sobre o momento da previsão e sobre o custo dos erros | 10 |
| **2. Análise exploratória** | Feita somente no treino; gráficos pertinentes; conclusões que **embasam** as decisões seguintes | 15 |
| **3. Pré-processamento e vazamento** | Tratamento adequado de ausentes, escala e categorias; ajuste apenas no treino; remoção justificada de colunas | 20 |
| **4. Modelagem** | Baseline; ao menos três modelos comparados; escolha justificada do modelo final | 15 |
| **5. Validação, overfitting e regularização** | Validação cruzada; análise treino × validação; efeito da regularização bem interpretado | 20 |
| **6. Métricas e avaliação final** | Métrica adequada ao problema; desbalanceamento tratado (se houver); teste usado corretamente; análise de erros | 10 |
| **7. Conclusões e comunicação** | Respostas claras e com as próprias palavras; limitações honestas; notebook organizado e executável | 10 |
| **Total** | | **100** |

**Penalidades importantes:**
- Pré-processamento ajustado no dataset completo, ou decisões tomadas com base no teste (vazamento de informação): perda de até **50%** dos critérios 3 e 6.
- Notebook que não executa do início ao fim: perda de até **20 pontos**.
- Uso de coluna que vaza o alvo sem discussão: perda de até **50%** dos critérios 1 e 3.

# Apêndice C — Sugestões de datasets

Qualquer dataset é aceito, desde que seja um problema de **regressão** ou **classificação binária** com pelo menos **500 linhas** e **5 features**. Algumas sugestões:

**Regressão**
- *California Housing* — preço mediano de imóveis por região (`sklearn.datasets.fetch_california_housing`).
- *Medical Cost Personal Datasets* (Kaggle) — custo de plano de saúde a partir de idade, IMC, tabagismo etc.
- *Wine Quality* (UCI) — nota de qualidade do vinho a partir de propriedades químicas.
- *Bike Sharing* (UCI) — bicicletas alugadas por hora/dia (atenção à ordem temporal!).
- *Student Performance* (UCI) — nota final de alunos (atenção às notas parciais, que podem vazar o alvo).

**Classificação binária**
- *Telco Customer Churn* (Kaggle) — cliente cancelou ou não o serviço (classes desbalanceadas).
- *Bank Marketing* (UCI) — cliente aderiu ou não a uma campanha (desbalanceado; atenção à coluna `duration`, que vaza o alvo).
- *Adult / Census Income* (UCI) — renda anual acima ou abaixo de 50 mil dólares.
- *Pima Indians Diabetes* (Kaggle/UCI) — diagnóstico de diabetes (atenção a zeros que representam valores ausentes).

**Dados públicos brasileiros**
- **IBGE**, **dados.gov.br**, **INEP** (microdados do ENEM) e **DATASUS** têm bases que podem ser adaptadas — ex.: prever a nota do ENEM (regressão) ou se um estudante atinge uma nota de corte (classificação). Bases grandes podem ser amostradas.

**Dados sintéticos**
- É permitido usar dados gerados com IA ou com `sklearn.datasets.make_regression` / `make_classification`, **desde que** você descreva como foram gerados e discuta a limitação de não serem dados reais.